In [1]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [2]:
class TrackedObject:
  def __init__(self, **kwargs):
    # history сам добавляем через super().__setattr__, чтобы не ловить в логе
    super().__setattr__('history', [])
    for name, value in kwargs.items():
      setattr(self, name, value)

  def __setattr__(self, name, value):
    before_size = len(self.__dict__)
    existed_before = name in self.__dict__
    old_value = self.__dict__.get(name,"<missing>")

    super().__setattr__(name, value)
    after_size = len(self.__dict__)

    self.history.append({
        "op": "set",
        "name": name,
        "old_value": old_value,
        "new_value": value,
        "existed_before": existed_before,
        "before_size": before_size,
        "after_size": after_size,
    })
  def __delattr__(self, name):
    before_size = len(self.__dict__)
    existed_before = name in self.__dict__

    if not existed_before:
        raise AttributeError(f"{type(self).__name__!r} object has no attribute {name!r}")

    old_value = self.__dict__[name]

    super().__delattr__(name)

    after_size = len(self.__dict__)

    self.history.append({
        "op": "del",
        "name": name,
        "old_value": old_value,
        "existed_before": existed_before,
        "dict_size_before": before_size,
        "dict_size_after": after_size,
    })
obj = TrackedObject(x=1, y=2)
obj.z = 3                # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x

print(obj.__dict__)
for event in obj.history:
  print(event)

{'history': [{'op': 'set', 'name': 'x', 'old_value': '<missing>', 'new_value': 1, 'existed_before': False, 'before_size': 1, 'after_size': 2}, {'op': 'set', 'name': 'y', 'old_value': '<missing>', 'new_value': 2, 'existed_before': False, 'before_size': 2, 'after_size': 3}, {'op': 'set', 'name': 'z', 'old_value': '<missing>', 'new_value': 3, 'existed_before': False, 'before_size': 3, 'after_size': 4}, {'op': 'set', 'name': '__str__', 'old_value': '<missing>', 'new_value': <function <lambda> at 0x7b3bca1d5da0>, 'existed_before': False, 'before_size': 4, 'after_size': 5}, {'op': 'del', 'name': 'x', 'old_value': 1, 'existed_before': True, 'dict_size_before': 5, 'dict_size_after': 4}], 'y': 2, 'z': 3, '__str__': <function <lambda> at 0x7b3bca1d5da0>}
{'op': 'set', 'name': 'x', 'old_value': '<missing>', 'new_value': 1, 'existed_before': False, 'before_size': 1, 'after_size': 2}
{'op': 'set', 'name': 'y', 'old_value': '<missing>', 'new_value': 2, 'existed_before': False, 'before_size': 2, 'aft

## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [3]:
def merge(seqs):
  result = []

  seqs = [list(seq) for seq in seqs if seq]
  while seqs:
    for seq in seqs:
      candidate = seq[0]
      if not any(candidate in other[1:] for other in seqs):
        break
    else:
      raise TypeError("C3 linearization can't be built")
    result.append(candidate)
    new_seqs = []
    for seq in seqs:
      if seq and seq[0] == candidate:
        seq.pop(0)
      if seq:
        new_seqs.append(seq)
    seqs = new_seqs

  return result

def c3_linearize(cls):
  if cls is object:
    return [object]
  bases = cls.__bases__
  base_linearizations = [c3_linearize(base) for base in bases]
  return [cls] + merge(base_linearizations + [list(bases)])

In [4]:
class Root:
    def who(self):
        return "Root"


class A(Root):
    def who(self):
        return "A"


class B(Root):
    def who(self):
        return "B"


class C(A, B):
    pass


class D(B):
    def who(self):
        return "D"


class E(A):
    pass


class F(C, D, E):
    pass

In [5]:
for cls in [Root, A, B, C, D, E, F]:
    my_mro = c3_linearize(cls)
    real_mro = list(cls.__mro__)

    print(f"{cls.__name__}:")
    print("  c3_linearize:", [c.__name__ for c in my_mro])
    print("  __mro__     :", [c.__name__ for c in real_mro])
    print()

Root:
  c3_linearize: ['Root', 'object']
  __mro__     : ['Root', 'object']

A:
  c3_linearize: ['A', 'Root', 'object']
  __mro__     : ['A', 'Root', 'object']

B:
  c3_linearize: ['B', 'Root', 'object']
  __mro__     : ['B', 'Root', 'object']

C:
  c3_linearize: ['C', 'A', 'B', 'Root', 'object']
  __mro__     : ['C', 'A', 'B', 'Root', 'object']

D:
  c3_linearize: ['D', 'B', 'Root', 'object']
  __mro__     : ['D', 'B', 'Root', 'object']

E:
  c3_linearize: ['E', 'A', 'Root', 'object']
  __mro__     : ['E', 'A', 'Root', 'object']

F:
  c3_linearize: ['F', 'C', 'D', 'E', 'A', 'B', 'Root', 'object']
  __mro__     : ['F', 'C', 'D', 'E', 'A', 'B', 'Root', 'object']



## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [6]:
class SecureBase:
    def __init__(self):
      self.__secret_value = "base secret"
      self._semi_private = "base semi"
      self.public = "base public"

class SecureChild(SecureBase):
    def __init__(self):
      super().__init__()
      self.__secret_value = "child secret"
      self._semi_private = "child semi"
      self.public = "child public"


    def dump_dict(self):
      return self.__dict__


base = SecureBase()
child = SecureChild()

print(f"Base Dict: {base.__dict__}\n")
print(f"Child Dict: {child.__dict__}\n")

print("Base dir fragment:")
print([name for name in dir(base) if "secret" in name or "private" in name or name == "public"])
print()

print("Child dir fragment:")
print([name for name in dir(child) if "secret" in name or "private" in name or name == "public"])
print()

print("Child dump_dict():")
print(child.dump_dict())
print()

print("Access mangled from outside:")
print("base._SecureBase__secret_value =", base._SecureBase__secret_value)
print("child._SecureBase__secret_value =", child._SecureBase__secret_value)
print("child._SecureChild__secret_value =", child._SecureChild__secret_value)

Base Dict: {'_SecureBase__secret_value': 'base secret', '_semi_private': 'base semi', 'public': 'base public'}

Child Dict: {'_SecureBase__secret_value': 'base secret', '_semi_private': 'child semi', 'public': 'child public', '_SecureChild__secret_value': 'child secret'}

Base dir fragment:
['_SecureBase__secret_value', '_semi_private', 'public']

Child dir fragment:
['_SecureBase__secret_value', '_SecureChild__secret_value', '_semi_private', 'public']

Child dump_dict():
{'_SecureBase__secret_value': 'base secret', '_semi_private': 'child semi', 'public': 'child public', '_SecureChild__secret_value': 'child secret'}

Access mangled from outside:
base._SecureBase__secret_value = base secret
child._SecureBase__secret_value = base secret
child._SecureChild__secret_value = child secret


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [9]:
class NoSlots:
  def __init__(self, x, y):
    self.x = x
    self.y = y

class WithSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
      self.x = x
      self.y = y

class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")
    def __init__(self, x):
      self.x = x

def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = None
        d["dict_keys"] = None
    return d

n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

print("Before Dynamic attributes:")
print(describe_instance(n))
print(describe_instance(w))
print(describe_instance(ww))
# Попытка динамического добавления атрибутов
print("Dynamic attrubites:")
for obj in [n, w, ww]:
    try:
        obj.z = 999
        print(f"{type(obj).__name__}: z added successfully")
    except AttributeError as e:
        print(f"{type(obj).__name__}: cannot add z -> {e}")

print()

print("After dynamic attrs:")
print(describe_instance(n))
print(describe_instance(w))
print(describe_instance(ww))
print()

# weakref
print("Weakref test:")

for obj in [n, w, ww]:
    try:
        r = weakref.ref(obj)
        print(f"{type(obj).__name__}: weakref created, r() ->", r())
    except TypeError as e:
        print(f"{type(obj).__name__}: weakref not supported -> {e}")

Before Dynamic attributes:
{'type': 'NoSlots', 'has_dict': True, 'has_weakref': True, 'size_obj': 48, 'size_dict': 296, 'dict_keys': ['x', 'y']}
{'type': 'WithSlots', 'has_dict': False, 'has_weakref': False, 'size_obj': 48, 'size_dict': None, 'dict_keys': None}
{'type': 'WithSlotsWeak', 'has_dict': False, 'has_weakref': True, 'size_obj': 56, 'size_dict': None, 'dict_keys': None}
Dynamic attrubites:
NoSlots: z added successfully
WithSlots: cannot add z -> 'WithSlots' object has no attribute 'z'
WithSlotsWeak: cannot add z -> 'WithSlotsWeak' object has no attribute 'z'

After dynamic attrs:
{'type': 'NoSlots', 'has_dict': True, 'has_weakref': True, 'size_obj': 48, 'size_dict': 296, 'dict_keys': ['x', 'y', 'z']}
{'type': 'WithSlots', 'has_dict': False, 'has_weakref': False, 'size_obj': 48, 'size_dict': None, 'dict_keys': None}
{'type': 'WithSlotsWeak', 'has_dict': False, 'has_weakref': True, 'size_obj': 56, 'size_dict': None, 'dict_keys': None}

Weakref test:
NoSlots: weakref created, r()

## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [15]:
import sys

class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name
        if parent is None:
          self.parent = None
        elif weak_parent:
          self.parent = weakref.ref(parent)
        else:
          self.parent = parent

    def get_parent(self):
        if self.parent is None:
          return None
        if isinstance(self.parent, weakref.ReferenceType):
          return self.parent()
        return self.parent
    def __repr__(self):
      return f"Node ({self.name})"
def refcount(obj):
    return sys.getrefcount(obj) - 1

print("=== Strong cycle ===")
gc.collect()

a = Node("a")
b = Node("b", parent=a, weak_parent=False)

a.child = b

print("refcount(a):", refcount(a))
print("refcount(b):", refcount(b))
print("a.get_parent():", a.get_parent())
print("b.get_parent():", b.get_parent())

wa = weakref.ref(a)
wb = weakref.ref(b)

del a
del b

print("After del, before gc:")
print("wa() ->", wa())
print("wb() ->", wb())

collected = gc.collect()
print("gc.collect() collected:", collected)

print("After gc:")
print("wa() ->", wa())
print("wb() ->", wb())
print()

print("=== Weak parent reference ===")
gc.collect()

p = Node("parent")
c = Node("child", parent=p, weak_parent=True)

p.child = c

print("refcount(p):", refcount(p))
print("refcount(c):", refcount(c))
print("c.get_parent():", c.get_parent())

wp = weakref.ref(p)
wc = weakref.ref(c)

del p

print("After deleting p:")
print("wp() ->", wp())
print("wc() ->", wc())

if wc() is not None:
    print("wc().get_parent() ->", wc().get_parent())

del c

collected = gc.collect()
print("gc.collect() collected:", collected)
print("After deleting c:")
print("wp() ->", wp())
print("wc() ->", wc())

=== Strong cycle ===
refcount(a): 3
refcount(b): 3
a.get_parent(): None
b.get_parent(): Node (a)
After del, before gc:
wa() -> Node (a)
wb() -> Node (b)
gc.collect() collected: 2
After gc:
wa() -> None
wb() -> None

=== Weak parent reference ===
refcount(p): 2
refcount(c): 3
c.get_parent(): Node (parent)
After deleting p:
wp() -> None
wc() -> Node (child)
wc().get_parent() -> None
gc.collect() collected: 0
After deleting c:
wp() -> None
wc() -> None
